# Data profiling of the bronze layer

Bronze holds the raw data exactly as it was loaded, so before cleaning anything in silver I look for problems that the earlier checks (row counts, rescued data, NULLs) cannot show. This notebook covers suspicious values, zero rows, extreme quantities and duplicates in `transaction_data`. The decisions for silver are collected at the end.

In [0]:
USE CATALOG dunnhumby;
USE SCHEMA bronze;

### Column types (transaction_data)
Do the inferred types fit the data?

In [0]:
DESCRIBE transaction_data

`inferSchema` assigned sensible types: keys and quantities are integers, amounts are doubles.

**Decisions for silver:**
- build a real date and time from `DAY` and `TRANS_TIME`, which are plain integers
- cast money columns (`SALES_VALUE`, `RETAIL_DISC`, `COUPON_DISC`, `COUPON_MATCH_DISC`) to `DECIMAL(10,2)`
- standardize column names to snake_case (`household_key` is lower case, the rest upper case)


### Transaction value ranges
What are the ranges of `QUANTITY` and `SALES_VALUE`, which zero combinations occur, and do those rows still carry a discount?

In [0]:
-- What are the minimum and maximum values, and how many rows are zero or negative?
SELECT
    MIN(QUANTITY) AS min_qty, MAX(QUANTITY) AS max_qty,
    MIN(SALES_VALUE) AS min_sales, MAX(SALES_VALUE) AS max_sales,
    SUM(CASE WHEN QUANTITY <= 0 THEN 1 ELSE 0 END) AS qty_zero_or_negative,
    SUM(CASE WHEN SALES_VALUE <= 0 THEN 1 ELSE 0 END) AS sales_zero_or_negative
FROM transaction_data

In [0]:
-- How many rows have a zero quantity, a zero sales value, or both, and do they still carry a discount?
SELECT 
    QUANTITY = 0 AS qty_zero,
    SALES_VALUE = 0 AS sales_zero,
    SUM(CASE WHEN RETAIL_DISC <> 0 OR COUPON_DISC <> 0 OR COUPON_MATCH_DISC <> 0 THEN 1 ELSE 0 END) AS row_with_discount,
    COUNT(*) AS cnt
    FROM transaction_data
    GROUP BY 1,2

There are no negative values in `QUANTITY` or `SALES_VALUE`. Discount columns are negative by design.

| QUANTITY = 0 | SALES_VALUE = 0 | Rows | With a discount |
|---|---|---|---|
| no | yes | 4,451 | 3,749 |
| yes | yes | 14,399 | 4,980 |
| yes | no | 67 | 14 |
| no | no | 2,576,815 | 1,310,692 |

- Positive quantity and zero sales value (4,451 rows): 3,749 carry a discount, which fits fully discounted or free items.
- Zero quantity and zero sales value with a discount (4,980 rows): most likely discount adjustment lines, not purchases.
- Zero quantity, zero sales value and no discount (14,399 minus 4,980 = 9,419 rows): no information at all.
- Zero quantity with a non-zero sales value (67 rows): they carry revenue, so they stay.

**Decision for silver:** drop only the 9,419 rows with zero quantity, zero sales value and no discount. Keep everything else and add expectations `QUANTITY >= 0` and `SALES_VALUE >= 0`.


### Extreme quantities
`QUANTITY` reaches almost 90,000, which cannot be a normal item count. Which products do these rows belong to?

In [0]:
-- Which products have the largest quantities?
SELECT PRODUCT_ID, QUANTITY
FROM transaction_data
ORDER BY QUANTITY DESC
LIMIT 20;

The 20 largest quantities come from only 3 products: 6534178 (16 rows), 6544236 (3 rows) and 6534166 (1 row), with quantities between 34,920 and 89,638. The queries below check what these products are.

In [0]:
-- What are the products with quantity above 1,000?
SELECT pr.PRODUCT_ID, pr.DEPARTMENT, pr.COMMODITY_DESC, pr.SUB_COMMODITY_DESC, COUNT(*) AS rows_over_1000, MAX(td.QUANTITY) AS max_qty
FROM transaction_data AS td
JOIN product AS pr ON td.PRODUCT_ID = pr.PRODUCT_ID
WHERE td.QUANTITY>1000
GROUP BY 1,2,3,4
ORDER BY rows_over_1000 DESC

In [0]:
-- Does anything other than gasoline have quantity above 1000?
SELECT pr.SUB_COMMODITY_DESC, COUNT(*) AS rows_over_1000
FROM transaction_data AS td
JOIN product AS pr ON td.PRODUCT_ID = pr.PRODUCT_ID
WHERE td.QUANTITY > 1000
GROUP BY pr.SUB_COMMODITY_DESC

In [0]:
-- How many gasoline products and rows are there?
SELECT
    COUNT(DISTINCT pr.PRODUCT_ID) AS fuel_products,
    COUNT(*) AS fuel_rows,
    ROUND(AVG(td.SALES_VALUE / NULLIF(td.QUANTITY, 0) * 1000), 2) AS avg_price_per_1000_units
FROM transaction_data AS td
JOIN product AS pr ON td.PRODUCT_ID = pr.PRODUCT_ID
WHERE pr.SUB_COMMODITY_DESC = 'GASOLINE-REG UNLEADED'

In [0]:
-- Does QUANTITY mean the same thing in all gasoline rows?
SELECT 
    CASE WHEN td.QUANTITY > 1000 THEN 'qty_over_1000' ELSE 'qty_up_to_1000' END AS grp,
    COUNT(*) AS fuel_rows,
    MIN(td.QUANTITY) AS min_qty,
    MAX(td.QUANTITY) AS max_qty,
    ROUND(percentile_approx(td.SALES_VALUE / nullif(td.QUANTITY, 0) * 1000 , 0.5),2 ) AS median_price_per_1000_units
FROM transaction_data AS td
JOIN product AS pr ON td.PRODUCT_ID = pr.PRODUCT_ID
WHERE pr.SUB_COMMODITY_DESC = 'GASOLINE-REG UNLEADED'
GROUP BY 1

All 23,101 rows with `QUANTITY` above 1,000 belong to gasoline (`SUB_COMMODITY_DESC = 'GASOLINE-REG UNLEADED'`). There are 35 gasoline products with 24,962 rows in total.

| Group | Rows | Quantity range | Median sales value per 1,000 units |
|---|---|---|---|
| quantity above 1,000 | 23,101 | 1,001 to 89,638 | 2.28 |
| quantity up to 1,000 | 1,861 | 1 to 1,000 | 15,000 |

In the first group the price fits thousandths of a gallon (1,000 units cost about 2.28). In the second group one unit costs about 15, which looks like a purchase recorded as a single item. So `QUANTITY` for fuel has two different meanings and cannot be used as an item count.

**Decision for silver:** keep all fuel rows, add an `is_fuel` flag based on the product (not on a quantity threshold) and keep the original `QUANTITY`. No upper bound expectation on `QUANTITY`. In gold, exclude fuel from "units sold" and unit price metrics.


### Duplicate check

`transaction_data` has no unique row id, so duplicates were checked as rows identical
across all business columns (audit columns and rows with QUANTITY = 0 excluded).

- Result: 0 duplicate groups.
- Repeat purchases of the same product in one basket are therefore not stored as
  identical rows, so there is nothing to deduplicate.

**Decision for silver:** no deduplication step is needed for `transaction_data`.

In [0]:
SELECT
    household_key, BASKET_ID, DAY, PRODUCT_ID, QUANTITY, SALES_VALUE,
    STORE_ID, RETAIL_DISC, TRANS_TIME, WEEK_NO, COUPON_DISC, COUPON_MATCH_DISC,
    COUNT(*) AS cnt
FROM transaction_data
WHERE QUANTITY > 0
GROUP BY ALL
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 20;

No duplicate groups found. Repeat purchases of the same product in one basket are not stored as identical rows, so there is nothing to deduplicate.

**Decision for silver:** no deduplication step for `transaction_data`.

## Findings and decisions for silver

| Table | Finding | Decision |
|---|---|---|
| transaction_data | `DAY` and `TRANS_TIME` are plain integers | Build a real date and time |
| transaction_data | Money columns are doubles | Cast to `DECIMAL(10,2)` |
| transaction_data | 9,419 rows with zero quantity, zero sales value and no discount | Drop them |
| transaction_data | `QUANTITY` for fuel has two different meanings | Add an `is_fuel` flag, keep the original value |
| transaction_data | No duplicate rows  | No deduplication step | 
| all tables | Inconsistent column name casing | Standardize to snake_case |